In [1]:
import numpy as np
from matplotlib import pyplot as plt

In [6]:
def di2xy_eqarea(dec, inc):
    if (inc == 90.0) or (inc == -90.0):
        x = y = 0.0
        return x, y 
    else:
        D = np.deg2rad(dec)
        I = np.deg2rad(inc)
        x = np.cos(I) * np.cos(D)
        y = np.cos(I) * np.sin(D)
        z = np.sin(I)
        L0 = 1.0 / np.sqrt(x**2 + y**2)
        L = L0 * np.sqrt(1.0 - np.abs(z))
        x *= L
        y *= L
        return x, y

In [ ]:
def plot_net(fignum=None, tick_spacing=10, ax=None, label_directions=False,
             label_type='cardinal'):
    """
    Draws circle and tick marks for equal area projection.

    Parameters:
        fignum: int or None
            Figure number to use for creating a new figure if no axis is provided.
        tick_spacing: int
            Interval for declination tick marks, default is 10.
        ax: matplotlib.axes.Axes or None
            Axis to plot on. If None, the current axis will be used (or created if fignum is given).
        label_directions: bool
            If True, label the directions around the perimeter of the net.
            Default is False.
        label_type: str
            Style of perimeter labels used when label_directions is True.
            Options are:
                'cardinal' (default): N, E, S, W
                'numeric': 000, 090, 180, 270
                'numeric_degree': 0°, 90°, 180°, 270°
                'slotz_special': 0° at the top (N) and 90° at the right (E),
                    with the bottom and left positions left blank.
    """
    if ax is None:
        if fignum is not None:
            plt.figure(num=fignum)
            plt.clf()
        ax = plt.gca()
    ax.axis("off")
    Dcirc = np.arange(0, 361.0)
    Icirc = np.zeros(361, dtype=float)
    Xcirc, Ycirc = [], []
    for k in range(361):
        XY = pmag.dimap(Dcirc[k], Icirc[k])
        Xcirc.append(XY[0])
        Ycirc.append(XY[1])
    ax.plot(Xcirc, Ycirc, "k")

    # Put on the tick marks
    Xsym, Ysym = [], []
    for I in range(tick_spacing, 100, tick_spacing):
        XY = pmag.dimap(0.0, I)
        Xsym.append(XY[0])
        Ysym.append(XY[1])
    ax.scatter(Xsym, Ysym, color="black", marker="_", s=10)

    Xsym, Ysym = [], []
    for I in range(tick_spacing, 100, tick_spacing):
        XY = pmag.dimap(90.0, I)
        Xsym.append(XY[0])
        Ysym.append(XY[1])
    ax.scatter(Xsym, Ysym, color="black", marker="|", s=10)

    Xsym, Ysym = [], []
    for I in range(tick_spacing, 90, tick_spacing):
        XY = pmag.dimap(180.0, I)
        Xsym.append(XY[0])
        Ysym.append(XY[1])
    ax.scatter(Xsym, Ysym, color="black", marker="_", s=10)

    Xsym, Ysym = [], []
    for I in range(tick_spacing, 90, tick_spacing):
        XY = pmag.dimap(270.0, I)
        Xsym.append(XY[0])
        Ysym.append(XY[1])
    ax.scatter(Xsym, Ysym, color="black", marker="|", s=10)

    for D in range(0, 360, tick_spacing):
        Xtick, Ytick = [], []
        for I in range(4):
            XY = pmag.dimap(D, I)
            Xtick.append(XY[0])
            Ytick.append(XY[1])
        ax.plot(Xtick, Ytick, "k")
    ax.set_aspect("equal")
    if label_directions:
        label_sets = {
            'cardinal': ("N", "E", "S", "W"),
            'numeric': ("000", "090", "180", "270"),
            'numeric_degree': (r"  0$\degree$", r" 90$\degree$",
                               r" 180$\degree$", r" 270$\degree$"),
            'slotz_special': (r"  0$\degree$", r" 90$\degree$", "", ""),
        }
        if label_type not in label_sets:
            raise ValueError(
                "label_type must be one of {}".format(sorted(label_sets)))
        north, east, south, west = label_sets[label_type]
        offset = 0.05
        fontsize = 12
        ax.text(0, 1 + offset, north, ha="center", va="bottom", fontsize=fontsize)
        ax.text(1 + offset, 0, east, ha="left", va="center", fontsize=fontsize)
        ax.text(0, -1 - offset, south, ha="center", va="top", fontsize=fontsize)
        ax.text(-1 - offset, 0, west, ha="right", va="center", fontsize=fontsize)
        ax.axis((-1.1, 1.1, -1.1, 1.1))
    else:
        ax.axis((-1.05, 1.05, -1.05, 1.05))

In [ ]:
def plot_di(dec, inc, color='k', marker='o', markersize=20, label=''):
    """
    Plot declination, inclination data on an equal area plot.

    Before this function is called a plot needs to be initialized with code that looks
    something like:
    >fignum = 1
    >plt.figure(num=fignum,figsize=(10,10),dpi=160)
    >ipmag.plot_net(fignum)

    Parameters:
        dec : declination being plotted
        inc : inclination being plotted
        color : the default color is black. Other colors can be chosen (e.g. 'r')
        marker : the default marker is a circle ('o')
        markersize : default size is 20
        label : the default label is blank ('')
    """
    X_down = []
    X_up = []
    Y_down = []
    Y_up = []
    color_down = []
    color_up = []

    if np.ndim(dec) == 0:
        # scalar dec/inc values
        XY = dimap(dec, inc)
        if inc >= 0:
            X_down.append(XY[0])
            Y_down.append(XY[1])
            color_down.append(color)
        else:
            X_up.append(XY[0])
            Y_up.append(XY[1])
            color_up.append(color)
    else:
        for n in range(len(dec)):
            XY = dimap(dec[n], inc[n])
            if inc[n] >= 0:
                X_down.append(XY[0])
                Y_down.append(XY[1])
                if isinstance(color, list):
                    color_down.append(color[n])
                else:
                    color_down.append(color)
            else:
                X_up.append(XY[0])
                Y_up.append(XY[1])
                if isinstance(color, list):
                    color_up.append(color[n])
                else:
                    color_up.append(color)

    if len(X_up) > 0:
        if connect_points:
            plt.plot(X_up, Y_up, ls = '-', linewidth=lw, color =lc, alpha = la, zorder=1)
        plt.scatter(X_up, Y_up, facecolors='none', edgecolors=color_up,
                    s=markersize, marker=marker, label=label,alpha=alpha, zorder=zorder)
    if len(X_down) > 0:
        if connect_points:
            plt.plot(X_down, Y_down, ls = '-', linewidth=lw, color =lc, alpha = la, zorder=1)
        plt.scatter(X_down, Y_down, facecolors=color_down, edgecolors=edge,
                    s=markersize, marker=marker, label=label,alpha=alpha, zorder=zorder)
    if legend == 'yes':
        plt.legend(loc=2)
    plt.tight_layout()
    if title is not None:
        plt.title(title)